In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'figure.titlesize': 22,
    'font.weight': 'normal',
    'axes.titleweight': 'bold'
})

# Basic operations

### Paths

In [ ]:
folder = "/content/drive/My Drive/volleypred/plots"

if not os.path.exists(folder):
    os.makedirs(folder)

In [ ]:
data_path = '/content/drive/My Drive/volleypred/data/'
plot_path = '/content/drive/My Drive/volleypred/plots/'

### Load the preprocessed dataset

In [ ]:
df = pd.read_csv(data_path + 'final_df.csv')
df.head()

### Divide statistics

In [ ]:
points_cols = ['Diff_Sum', 'Diff_BP', 'Diff_Ratio', 'Diff_Days_since_last_match']
serve_cols = [col for col in df.columns if 'Srv' in col]
reception_cols = [col for col in df.columns if 'Rec' in col]
attack_cols = [col for col in df.columns if 'Att' in col]
block_cols = [col for col in df.columns if 'Blk' in col]

numeric_cols = points_cols + serve_cols + reception_cols + attack_cols + block_cols
len(numeric_cols)

# Descriptive analysis

### Mean/std/min/max/quartiles

In [ ]:
df[numeric_cols].describe().T

### Skewness/variance

In [ ]:
df[numeric_cols].var()
df[numeric_cols].skew()

### Categorical values

In [ ]:
df['Season'].describe(include='object').T

### Unique values

In [ ]:
df.nunique()

# Categorical features analysis

### Clubs presence across seasons

In [ ]:
t1 = df[['Team_T1', 'Season']].rename(columns={'Team_T1': 'Team'})
t2 = df[['Team_T2', 'Season']].rename(columns={'Team_T2': 'Team'})
team_presence = pd.concat([t1, t2]).drop_duplicates()

presence_matrix = pd.crosstab(team_presence['Team'], team_presence['Season'])
presence_matrix = (presence_matrix > 0).astype(int)

presence_matrix['Total'] = presence_matrix.sum(axis=1)
presence_matrix = presence_matrix.sort_values('Total', ascending=False).drop(columns='Total')

plt.figure(figsize=(16, 12))
sns.heatmap(presence_matrix,
            cmap="Blues",
            cbar=False,
            linewidths=.5,
            linecolor='lightgrey')

plt.xlabel('Season')
plt.ylabel('Club')
plt.xticks(rotation=45)
plt.savefig(plot_path + 'club_presence.png', dpi=300)
plt.show()
print("The plot has been saved!")

### Teams per season

In [ ]:
teams_per_season = team_presence.groupby('Season')['Team'].nunique().reset_index()
teams_per_season.columns = ['Season', 'Team_Count']

plt.figure(figsize=(12, 6))
bars = plt.bar(teams_per_season['Season'], teams_per_season['Team_Count'], color='midnightblue', alpha=0.7)

for bar in bars:
    y_val = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, y_val + 0.2, int(y_val),
             ha='center', va='bottom')

plt.xlabel('Season')
plt.ylabel('Number of Teams')
plt.xticks(rotation=45)

plt.ylim(0, teams_per_season['Team_Count'].max() + 2)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.savefig(plot_path + 'teams_per_season.png', dpi=300)
plt.show()
print("The plot has been saved!")

Incorrect numbers of clubs in 2018/2019 and 2021/2022 seasons.

### Detection

In [ ]:
chosen_seasons = ['2018/2019', '2021/2022']

for season in chosen_seasons:
    season_df = df[df['Season'] == season]
    teams = pd.concat([season_df['Team_T1'], season_df['Team_T2']]).unique()

    print(f"Teams in season {season} ({len(teams)} teams):")
    print(sorted(teams))
    print("-" * 30)

Research:
- Stocznia Szczecin left the league due to bankrupcy
- 2021/2022 has to be checked

In [ ]:
bedzin_21_22 = df[
    (df['Season'] == '2021/2022') &
    ((df['Team_T1'] == 'MKS Będzin') | (df['Team_T2'] == 'MKS Będzin'))
]

print(f"{len(bedzin_21_22)} matches found for MKS Będzin in the 2021/2022 season.")
display(bedzin_21_22[['Date', 'Team_T1', 'Team_T2', 'Score']])

The dataset includes a play-off match between a team from the 1st league and a team from the PlusLiga

### Matches per season

In [ ]:
matches_per_season = df.groupby('Season').size().reset_index(name='Match_Count')

plt.figure(figsize=(12, 6))
bars = plt.bar(matches_per_season['Season'], matches_per_season['Match_Count'], color='cadetblue', alpha=0.8)

for bar in bars:
    y_val = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, y_val + 2, int(y_val),
             ha='center', va='bottom')

plt.xlabel('Season')
plt.ylabel('Number of Matches')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.savefig(plot_path + 'matches_per_season.png', dpi=300)
plt.show()
print("The plot has been saved!")

The dataset does not include all matches.

### Score distribution

In [ ]:
plt.figure(figsize=(10, 6))

order = ['3:0', '3:1', '3:2', '2:3', '1:3', '0:3']
result_counts = df['Score'].value_counts().reindex(order)

barplot = sns.barplot(
    x=result_counts.index,
    y=result_counts.values,
    hue=result_counts.index,
    palette='coolwarm'
)

for i, value in enumerate(result_counts.values):
    barplot.text(i, value + 5, int(value),
                 color='black', ha="center", va='bottom')

plt.xlabel('Result')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)

plt.axvline(x=2.5, color='red', linestyle='--', label='Win/Loss Boundary')
plt.legend()

plt.tight_layout()
plt.savefig(plot_path + 'score_distribution.png', dpi=300)
plt.show()

print("The plot has been saved!")

### Target distribution by season

In [ ]:
plt.figure(figsize=(10, 6))
result_counts = pd.crosstab(df['Season'], df['Score'])

desired_order = ['3:0', '3:1', '3:2', '2:3', '1:3', '0:3']
result_dist = result_counts.reindex(columns=desired_order)

ax = result_dist.plot(kind='bar', stacked=True, figsize=(14, 8), colormap='coolwarm', edgecolor='white')

for i, (idx, row) in enumerate(result_dist.iterrows()):
    total = row.sum()
    plt.text(i, total + 2, int(total), ha='center')

plt.xlabel('Season')
plt.ylabel('Number of Matches')
plt.xticks(rotation=45)

plt.legend(title='Match Result (Sets)', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()

plt.savefig(plot_path + 'result_distribution_per_season.png', dpi=300)
plt.show()

print(f"Plot saved to: {plot_path}")


# Numerical features

### Boxplots

In [ ]:
def generate_boxplot(cols, n_rows, n_cols,
                     file_name='Boxplot'):

    score_order = ['3:0', '3:1', '3:2', '2:3', '1:3', '0:3']

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 4))

    if n_rows * n_cols > 1:
        axes_flat = axes.ravel()
    else:
        axes_flat = [axes]

    for idx, col in enumerate(cols):
        sns.boxplot(
            data=df,
            x='Score',
            y=col,
            order=score_order,
            hue='Score',
            hue_order=score_order,
            ax=axes_flat[idx],
            palette='Set2',
            legend=False
        )

        axes_flat[idx].set_title(f'{col} by Result')
        axes_flat[idx].set_xlabel('Match Result')
        axes_flat[idx].set_ylabel(col)

        axes_flat[idx].tick_params(axis='x', rotation=45)

    for i in range(idx + 1, n_rows * n_cols):
        fig.delaxes(axes_flat[i])

    plt.tight_layout()

    save_path = plot_path + file_name
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Plot saved to: {save_path}")

In [ ]:
generate_boxplot(points_cols, 1, 4, file_name='boxplot_points.png')

In [ ]:
generate_boxplot(serve_cols, 1, 4, file_name='boxplot_serve.png')

In [ ]:
generate_boxplot(reception_cols, 1, 4, file_name='boxplot_reception.png')

In [ ]:
generate_boxplot(attack_cols, 2, 3, file_name='boxplot_attack.png')

In [ ]:
generate_boxplot(block_cols, 1, 3, file_name='boxplot_block.png')

### Histograms + kde

In [ ]:
def generate_histograms(cols, n_rows, n_cols,
                        file_name='histplot.png'):

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 4))

    if n_rows * n_cols > 1:
        axes_flat = axes.ravel()
    else:
        axes_flat = [axes]


    for idx, col in enumerate(cols):
        axes_flat[idx].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        axes_flat[idx].set_title(f'{col}')
        axes_flat[idx].set_xlabel(col)
        axes_flat[idx].set_ylabel('Frequency')

        ax2 = axes_flat[idx].twinx()
        df[col].plot(kind='kde', ax=ax2, color='navy', linewidth=2, legend=False)
        ax2.set_ylabel('Density')

        ax2.yaxis.label.set_color('navy')
        ax2.tick_params(axis='y', labelcolor='navy')

    for i in range(idx + 1, n_rows * n_cols):
        fig.delaxes(axes_flat[i])

    plt.tight_layout()

    save_path = plot_path + file_name
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Plot saved to: {save_path}")

In [ ]:
generate_histograms(points_cols, 1, 4, file_name='histplot_points.png')

In [ ]:
generate_histograms(serve_cols, 1, 4, file_name='histplot_serve.png')

In [ ]:
generate_histograms(reception_cols, 1, 4, file_name='histplot_reception.png')

In [ ]:
generate_histograms(attack_cols, 2, 3, file_name='histplot_attack.png')

In [ ]:
generate_histograms(block_cols, 1, 3, file_name='histplot_block.png')

### Corelation matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
correlation_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))

sns.heatmap(correlation_matrix,
            annot=True,
            fmt='.2f',
            cmap='vlag',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})

plt.xlabel('')
plt.ylabel('')
plt.tight_layout()
plt.savefig(plot_path + 'correlation_matrix.png', dpi=300)
plt.show()
print(f"Plot saved to: {plot_path}")

In [ ]:
print("\nHighest correlations (excluding self-correlations):")
print("=" * 50)

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)
high_corr = correlation_matrix.where(mask)

corr_pairs = high_corr.unstack().dropna().sort_values(ascending=False)
print(corr_pairs[corr_pairs < 1.0].head(10))